# 24 Local Topic Extraction / Candidate Generation

This notebook performs deterministic, local-only candidate extraction from Phase 23 prepared Bluesky posts.

- No Snowflake access
- No reruns of firehose/hydration/enrichment
- No final matching in this phase


**Notebook purpose:** Extracts topic candidates (hashtags + n-grams) from Phase 23 prepared Bluesky posts for downstream trend matching. Produces a candidate-per-post parquet.

**Required data:** `local/derived/bluesky/bluesky_posts_prepared.parquet` (from notebook 23) and `local/reference_snapshots/twitter_trending/twitter_trending_normalized.parquet` (from notebook 22).

**Run order:** Run after notebook 23 (text prep). Run before notebook 25 (trend matching).

## 1. Load Inputs and Validate Schema

Use the actual prepared Bluesky schema from Phase 23 and fail fast if required fields are missing.


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd


def find_repo_root() -> Path:
    probe = Path.cwd().resolve()
    for candidate in [probe, *probe.parents]:
        if (candidate / 'src' / 'nlp' / 'topic_candidate_generation.py').exists():
            return candidate
    raise FileNotFoundError('Could not find repo root containing src/nlp/topic_candidate_generation.py')


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.nlp.topic_candidate_generation import (
    REQUIRED_PREPARED_COLUMNS,
    generate_topic_candidates_dataframe,
    summarize_topic_candidates,
)

prepared_path = ROOT / "local/derived/bluesky/bluesky_posts_prepared.parquet"
twitter_ref_path = ROOT / "local/reference_snapshots/twitter_trending/twitter_trending_normalized.parquet"

if not prepared_path.exists():
    raise FileNotFoundError(
        "DATA NOT YET AVAILABLE -- run notebook 23 (Bluesky text preparation) first.\n"
        f"Missing: {prepared_path}"
    )

prepared_df = pd.read_parquet(prepared_path)
print("Prepared rows:", len(prepared_df))
print("Prepared columns:", len(prepared_df.columns))
print("Twitter normalized reference exists:", twitter_ref_path.exists())

missing = [c for c in REQUIRED_PREPARED_COLUMNS if c not in prepared_df.columns]
if missing:
    raise ValueError(f"Missing required prepared columns: {missing}")

prepared_df[REQUIRED_PREPARED_COLUMNS].head(5)

Prepared rows: 19999
Prepared columns: 26
Twitter normalized reference exists: True


,uri,post_created_at,post_text_raw,post_text_clean,post_text_alnum
0,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,2026-04-02T18:16:23.346Z,when he does finally die there'll arise a new ...,when he does finally die there'll arise a new ...,when he does finally die there ll arise a new ...
1,at://did:plc:222p42fegwhwfyrc3gqam76j/app.bsky...,2026-04-02T18:13:37.972Z,What could Bondi have expected? The DOW is bel...,what could bondi have expected the dow is belo...,what could bondi have expected the dow is belo...
2,at://did:plc:223eaz6uecpa3t63jdlojutp/app.bsky...,2026-04-02T18:14:16.501Z,An important subject & obviously well done. Wi...,an important subject obviously well done wishi...,an important subject obviously well done wishi...
3,at://did:plc:22auflhdbiemhxvkgvpusmmd/app.bsky...,2026-04-02T18:15:23.338Z,i never flirted before.. i just exist or look ...,i never flirted before i just exist or look at...,i never flirted before i just exist or look at...
4,at://did:plc:22bb4y5cxdp3eucibibb4fkp/app.bsky...,2026-04-02T18:16:39.167Z,Gonna be slow posting for a few days. Out on v...,gonna be slow posting for a few days out on va...,gonna be slow posting for a few days out on va...


In [2]:
prepared_df.dtypes.to_frame("dtype")


,dtype
uri,str
post_created_at,str
source_run_tag,str
text_source,str
raw_capture_run_id,str
raw_captured_at,str
raw_repo_did,str
raw_record_created_at,str
hydrated_capture_run_id,str
hydrated_hydrate_run_id,str


## 2. Run Deterministic Candidate Extraction

Balanced extraction strategy: hashtag extraction + filtered 2/3/4-grams + conservative unigram fallback.


In [3]:
candidates_df = generate_topic_candidates_dataframe(prepared_df, max_candidates_per_post=20)
candidates_df = candidates_df.sort_values(["uri", "candidate_rank_in_post"], kind="stable").reset_index(drop=True)
print("Candidate rows:", len(candidates_df))
print("Posts with candidates:", candidates_df["uri"].nunique())
candidates_df.head(10)


Candidate rows: 237351
Posts with candidates: 17958


,uri,post_created_at,post_text_raw,post_text_clean,post_text_alnum,candidate_phrase_raw,candidate_phrase_clean,candidate_phrase_alnum,candidate_phrase_no_hash,candidate_source_type,candidate_token_count,candidate_char_count,candidate_rank_in_post,is_hashtag_candidate,contains_digit,is_unigram_fallback,candidate_start_index
0,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,2026-04-02T18:16:23.346Z,when he does finally die there'll arise a new ...,when he does finally die there'll arise a new ...,when he does finally die there ll arise a new ...,does finally die there,does finally die there,does finally die there,does finally die there,ngram_4,4,22,1,False,False,False,2
1,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,2026-04-02T18:16:23.346Z,when he does finally die there'll arise a new ...,when he does finally die there'll arise a new ...,when he does finally die there ll arise a new ...,finally die there ll,finally die there ll,finally die there ll,finally die there ll,ngram_4,4,20,2,False,False,False,3
2,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,2026-04-02T18:16:23.346Z,when he does finally die there'll arise a new ...,when he does finally die there'll arise a new ...,when he does finally die there ll arise a new ...,die there ll arise,die there ll arise,die there ll arise,die there ll arise,ngram_4,4,18,3,False,False,False,4
3,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,2026-04-02T18:16:23.346Z,when he does finally die there'll arise a new ...,when he does finally die there'll arise a new ...,when he does finally die there ll arise a new ...,ll arise a new,ll arise a new,ll arise a new,ll arise a new,ngram_4,4,14,4,False,False,False,6
4,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,2026-04-02T18:16:23.346Z,when he does finally die there'll arise a new ...,when he does finally die there'll arise a new ...,when he does finally die there ll arise a new ...,arise a new myth,arise a new myth,arise a new myth,arise a new myth,ngram_4,4,16,5,False,False,False,7
5,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,2026-04-02T18:16:23.346Z,when he does finally die there'll arise a new ...,when he does finally die there'll arise a new ...,when he does finally die there ll arise a new ...,new myth cycle about,new myth cycle about,new myth cycle about,new myth cycle about,ngram_4,4,20,6,False,False,False,9
6,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,2026-04-02T18:16:23.346Z,when he does finally die there'll arise a new ...,when he does finally die there'll arise a new ...,when he does finally die there ll arise a new ...,cycle about the president,cycle about the president,cycle about the president,cycle about the president,ngram_4,4,25,7,False,False,False,11
7,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,2026-04-02T18:16:23.346Z,when he does finally die there'll arise a new ...,when he does finally die there'll arise a new ...,when he does finally die there ll arise a new ...,president who waits resting,president who waits resting,president who waits resting,president who waits resting,ngram_4,4,27,8,False,False,False,14
8,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,2026-04-02T18:16:23.346Z,when he does finally die there'll arise a new ...,when he does finally die there'll arise a new ...,when he does finally die there ll arise a new ...,resting in his tomb,resting in his tomb,resting in his tomb,resting in his tomb,ngram_4,4,19,9,False,False,False,17
9,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,2026-04-02T18:16:23.346Z,when he does finally die there'll arise a new ...,when he does finally die there'll arise a new ...,when he does finally die there ll arise a new ...,until when the need,until when the need,until when the need,until when the need,ngram_4,4,19,10,False,False,False,21


## 3. Candidate Volume and Quality Profiling

Summarize coverage, source/type distribution, and posts with no candidates.


In [4]:
summary = summarize_topic_candidates(prepared_posts_df=prepared_df, candidates_df=candidates_df)
posts_with_no_candidates = sorted(set(prepared_df["uri"]) - set(candidates_df["uri"]))

summary


{'total_posts_processed': 19999,
 'posts_with_candidates': 17958,
 'posts_without_candidates': 2041,
 'total_candidate_rows': 237351,
 'avg_candidates_per_post_all': 11.868143407170358,
 'avg_candidates_per_post_with_candidates': 13.217006348145674,
 'candidate_source_type_counts': {'ngram_4': 129833,
  'ngram_3': 66523,
  'ngram_2': 34982,
  'hashtag': 4986,
  'unigram_fallback': 1027},
 'candidate_token_count_distribution': {'4': 129833,
  '3': 66527,
  '2': 35070,
  '1': 5921},
 'top_candidate_phrases': [{'candidate_phrase_alnum': 'words nine hundred seventy',
   'count': 245},
  {'candidate_phrase_alnum': 'nine hundred seventy five', 'count': 245},
  {'candidate_phrase_alnum': 'hundred seventy five thousand', 'count': 245},
  {'candidate_phrase_alnum': 'words nine hundred', 'count': 245},
  {'candidate_phrase_alnum': 'nine hundred seventy', 'count': 245},
  {'candidate_phrase_alnum': 'hundred seventy five', 'count': 245},
  {'candidate_phrase_alnum': 'seventy five thousand', 'count

In [5]:
pd.Series(posts_with_no_candidates, name="uri_with_no_candidates")


0       at://did:plc:22ezkuvas6f545oal47snp5x/app.bsky...
1       at://did:plc:22k3roaumaq77xwhhpdcjdka/app.bsky...
2       at://did:plc:22ximgct4dxbmv4hlekxmnmp/app.bsky...
3       at://did:plc:24kuxsyofab2uqg22jaux3g7/app.bsky...
4       at://did:plc:24yxw26xlpei33mv7k4iuwi6/app.bsky...
                              ...                        
2036    at://did:plc:zytoslewjqri7z3gmvsfun4s/app.bsky...
2037    at://did:plc:zytoslewjqri7z3gmvsfun4s/app.bsky...
2038    at://did:plc:zzififr6febsnsu754pidsxi/app.bsky...
2039    at://did:plc:zzt3q7qhhnxofqxiwaf37ztd/app.bsky...
2040    at://did:plc:zzvnxlcsrmydtb4led7a7ds7/app.bsky...
Name: uri_with_no_candidates, Length: 2041, dtype: str

## 4. Representative Examples

Inspect useful candidates, hashtag-derived candidates, and noisy/edge cases.


In [6]:
useful_examples = candidates_df[[
    "uri",
    "candidate_source_type",
    "candidate_phrase_raw",
    "candidate_phrase_alnum",
    "candidate_rank_in_post",
]].head(25)
useful_examples


,uri,candidate_source_type,candidate_phrase_raw,candidate_phrase_alnum,candidate_rank_in_post
0,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,ngram_4,does finally die there,does finally die there,1
1,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,ngram_4,finally die there ll,finally die there ll,2
2,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,ngram_4,die there ll arise,die there ll arise,3
3,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,ngram_4,ll arise a new,ll arise a new,4
4,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,ngram_4,arise a new myth,arise a new myth,5
5,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,ngram_4,new myth cycle about,new myth cycle about,6
6,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,ngram_4,cycle about the president,cycle about the president,7
7,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,ngram_4,president who waits resting,president who waits resting,8
8,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,ngram_4,resting in his tomb,resting in his tomb,9
9,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,ngram_4,until when the need,until when the need,10


In [7]:
hashtag_examples = candidates_df.loc[candidates_df["is_hashtag_candidate"], [
    "uri",
    "candidate_phrase_raw",
    "candidate_phrase_clean",
    "candidate_phrase_no_hash",
    "candidate_source_type",
]]
hashtag_examples


,uri,candidate_phrase_raw,candidate_phrase_clean,candidate_phrase_no_hash,candidate_source_type
99,at://did:plc:22dulstyhpc7jfpogwtyqkhv/app.bsky...,#nurse,#nurse,nurse,hashtag
100,at://did:plc:22dulstyhpc7jfpogwtyqkhv/app.bsky...,#scrubs,#scrubs,scrubs,hashtag
101,at://did:plc:22dulstyhpc7jfpogwtyqkhv/app.bsky...,#naughtynurse,#naughtynurse,naughtynurse,hashtag
102,at://did:plc:22dulstyhpc7jfpogwtyqkhv/app.bsky...,#publicflash,#publicflash,publicflash,hashtag
103,at://did:plc:22dulstyhpc7jfpogwtyqkhv/app.bsky...,#nurseboobs,#nurseboobs,nurseboobs,hashtag
...,...,...,...,...,...
236851,at://did:plc:zxv5it2icki42nh53gycdjsi/app.bsky...,#toronto,#toronto,toronto,hashtag
236852,at://did:plc:zxv5it2icki42nh53gycdjsi/app.bsky...,#cntower,#cntower,cntower,hashtag
237148,at://did:plc:zzbpvnwurzheabujcgftoyfp/app.bsky...,#parool,#parool,parool,hashtag
237168,at://did:plc:zzbpvnwurzheabujcgftoyfp/app.bsky...,#parool,#parool,parool,hashtag


In [8]:
noise_pattern = r"\b(?:macro|youtube|bsky|profile|shorts|com)\b"
noisy_examples = candidates_df.loc[
    candidates_df["candidate_phrase_alnum"].str.contains(noise_pattern, regex=True, na=False),
    ["uri", "candidate_source_type", "candidate_phrase_alnum", "candidate_rank_in_post"],
].head(20)
noisy_examples


,uri,candidate_source_type,candidate_phrase_alnum,candidate_rank_in_post
281,at://did:plc:22wukjqyibxviwqmi5vdabxu/app.bsky...,ngram_4,open substack com pub,1
282,at://did:plc:22wukjqyibxviwqmi5vdabxu/app.bsky...,ngram_4,substack com pub narativ,2
283,at://did:plc:22wukjqyibxviwqmi5vdabxu/app.bsky...,ngram_3,open substack com,3
284,at://did:plc:22wukjqyibxviwqmi5vdabxu/app.bsky...,ngram_3,substack com pub,4
285,at://did:plc:22wukjqyibxviwqmi5vdabxu/app.bsky...,ngram_3,com pub narativ,5
496,at://did:plc:24e6n6hmxhouhrgwnff7o5tv/app.bsky...,ngram_3,share biomerieux com,18
557,at://did:plc:24e6n6hmxhouhrgwnff7o5tv/app.bsky...,ngram_3,share biomerieux com,19
569,at://did:plc:24goyoimcui6dhxo2jgltp5u/app.bsky...,ngram_4,facebook com jerseytreesforlife posts,11
774,at://did:plc:256kd427doj7f4oyd3wmxs7v/app.bsky...,ngram_4,75 100 colorguesser com,14
949,at://did:plc:25xz4mmocebkvk6uqarcjfax/app.bsky...,ngram_4,nbcnews com politics 202,19


## 5. Write Required Outputs

Write full candidate parquet, sample parquet/csv, and optional compact summary JSON.


In [9]:
local_out = ROOT / "local/derived/bluesky"
sample_out = ROOT / "data/samples"
local_out.mkdir(parents=True, exist_ok=True)
sample_out.mkdir(parents=True, exist_ok=True)

full_out = local_out / "bluesky_topic_candidates.parquet"
sample_parquet_out = sample_out / "bluesky_topic_candidates_sample_1000.parquet"
sample_csv_out = sample_out / "bluesky_topic_candidates_sample_1000.csv"
summary_out = local_out / "bluesky_topic_candidate_summary.json"

candidates_df.to_parquet(full_out, index=False)
candidates_df.head(1000).to_parquet(sample_parquet_out, index=False)
candidates_df.head(1000).to_csv(sample_csv_out, index=False)

summary_payload = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "phase": "24_local_topic_extraction_candidate_generation",
    "input_paths": {
        "prepared_bluesky_posts": str(prepared_path),
        "twitter_trending_normalized_reference": str(twitter_ref_path),
    },
    "prepared_schema_columns": prepared_df.columns.tolist(),
    "prepared_row_count": int(len(prepared_df)),
    "candidate_schema_columns": candidates_df.columns.tolist(),
    "candidate_summary": summary,
    "posts_with_no_candidates": posts_with_no_candidates,
    "posts_with_no_candidates_count": int(len(posts_with_no_candidates)),
    "candidate_source_type_counts": {
        str(k): int(v) for k, v in candidates_df["candidate_source_type"].value_counts(dropna=False).items()
    },
    "candidate_token_count_distribution": {
        str(int(k)): int(v) for k, v in candidates_df["candidate_token_count"].value_counts(dropna=False).items()
    },
    "top_candidate_phrases": [
        {"candidate_phrase_alnum": str(k), "count": int(v)}
        for k, v in candidates_df["candidate_phrase_alnum"].value_counts(dropna=False).head(25).items()
    ],
    "output_paths": {
        "full_candidates_parquet": str(full_out),
        "sample_candidates_parquet": str(sample_parquet_out),
        "sample_candidates_csv": str(sample_csv_out),
    },
}
summary_out.write_text(json.dumps(summary_payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

print("Wrote:", full_out)
print("Wrote:", sample_parquet_out)
print("Wrote:", sample_csv_out)
print("Wrote:", summary_out)

Wrote: /Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/local/derived/bluesky/bluesky_topic_candidates.parquet
Wrote: /Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/data/samples/bluesky_topic_candidates_sample_1000.parquet
Wrote: /Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/data/samples/bluesky_topic_candidates_sample_1000.csv
Wrote: /Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/local/derived/bluesky/bluesky_topic_candidate_summary.json


## 6. Readiness Note

This output is candidate-generation only and is designed to feed Phase 25 post-to-trend matching.
